In [ ]:
import akshare as ak
import pandas as pd

# 设置 Pandas 显示格式，让表格好看点
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("环境准备就绪，正在向 AkShare 请求数据...")

# 获取厦工股份 (600815) 最近 30 个交易日的日线数据
# 自动复权：qfq (前复权)
df = ak.stock_zh_a_hist(symbol="600815", period="daily", start_date="20240101", adjust="qfq")

# 提取最后 5 天的数据看看
last_5_days = df.tail(5)

print("\n成功获取数据！厦工股份最近 5 个交易日行情：")
print(last_5_days[['日期', '开盘', '收盘', '最高', '最低', '成交量', '换手率']])

In [ ]:
import akshare as ak
import pandas as pd
import sqlite3
import time

# 1. 连接本地 SQLite 数据库（如果不存在会自动创建）
db_name = 'stock_quant.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

print("1. 数据库连接成功，正在创建表结构...")

# 2. 创建状态机表 (stock_pipeline)，用于管理股票当前处于哪个阶段
cursor.execute('''
    CREATE TABLE IF NOT EXISTS stock_pipeline (
        code TEXT PRIMARY KEY,
        name TEXT,
        status INTEGER DEFAULT 0,  -- 0:横盘, 1:试盘, 2:回踩, 3:突破, 4:废弃
        test_high REAL,            -- 试盘当日最高价
        entry_date TEXT,           -- 进入该状态的日期
        update_time TEXT           -- 最后更新时间
    )
''')
conn.commit()

print("2. 获取全市场A股代码列表...")
# 获取A股所有股票代码和名称
df_all_stocks = ak.stock_info_a_code_name()

# 【关键点】为了测试，我们先只取前 10 只股票！测试成功后，你再去掉限制跑全量。
test_stocks = df_all_stocks.head(10)
print(f"成功获取代码表，本次准备拉取 {len(test_stocks)} 只股票的历史数据进行测试。")

print("3. 开始批量拉取数据并写入数据库...")

# 设定我们要拉取的历史时间范围（比如过去半年：2023年10月1日至今）
start_date = "20231001"
success_count = 0

for index, row in test_stocks.iterrows():
    code = row['code']
    name = row['name']
    
    try:
        # 拉取单只股票的日线前复权数据
        df_hist = ak.stock_zh_a_hist(symbol=code, period="daily", start_date=start_date, adjust="qfq")
        
        if not df_hist.empty:
            # 增加'代码'列，方便在数据库中区分是哪只股票的数据
            df_hist['code'] = code 
            
            # 使用 pandas 的 to_sql 方法，极其方便地将数据追加写入数据库
            # 注意：表名叫 daily_k_line
            df_hist.to_sql(name='daily_k_line', con=conn, if_exists='append', index=False)
            
            # 同时，把这只股票初始化到我们的“流水线状态表”中，默认状态为 0 (Watching)
            cursor.execute('''
                INSERT OR IGNORE INTO stock_pipeline (code, name, status, update_time)
                VALUES (?, ?, 0, date('now'))
            ''', (code, name))
            conn.commit()
            
            success_count += 1
            print(f"[{success_count}/10] {name} ({code}) 数据已入库。")
            
        # 【极其重要】为了防止被新浪/东财等接口封杀你的IP，每次请求必须休息一下！
        time.sleep(0.5) 
        
    except Exception as e:
        print(f"拉取 {name} ({code}) 失败，错误原因：{e}")

# 收尾工作
conn.close()
print(f"\n✅ 测试建库任务完成！成功写入 {success_count} 只股票的历史数据。")
print(f"请检查你左侧的文件夹，是否多出了一个名为 '{db_name}' 的数据库文件。")

In [ ]:
import akshare as ak
import urllib3
import requests

# 禁用满屏的 SSL 警告
urllib3.disable_warnings()

print(f"当前 AkShare 版本: {ak.__version__}")
print("正在尝试连接东方财富服务器...")

try:
    # 增加一个 requests 的基础连通性测试
    test_resp = requests.get("https://push2his.eastmoney.com", verify=False, timeout=5)
    print(f"东方财富服务器基础连通性状态码: {test_resp.status_code}")
    
    print("正在拉取平安银行数据...")
    # 只请求 1 只股票，缩短时间范围到几天，看看能不能通
    df = ak.stock_zh_a_hist(symbol="000001", period="daily", start_date="20240301", adjust="qfq")
    print("\n✅ 成功啦！数据如下：")
    print(df.head())
    
except Exception as e:
    print(f"\n❌ 依然失败，报错详情：{e}")

In [ ]:
import akshare as ak

print("正在尝试通过网易(163)接口拉取数据，绕过东财 IP 限制...")
try:
    # 注意：网易接口的股票代码需要带上前缀 sh 或 sz
    # 000001 是深市，所以是 sz000001
    df_163 = ak.stock_zh_a_hist_163(symbol="sz000001", start_date="20240301", end_date="20240401")
    
    print("\n✅ 成功啦！网易接口数据如下：")
    print(df_163.head())
    
except Exception as e:
    print(f"\n❌ 网易接口也失败了，报错详情：{e}")

In [1]:
import akshare as ak
import baostock as bs
import pandas as pd
import sqlite3
import time

db_name = 'stock_quant.db'
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

print("1. 数据库连接成功，正在创建表结构...")
cursor.execute('''
    CREATE TABLE IF NOT EXISTS stock_pipeline (
        code TEXT PRIMARY KEY,
        name TEXT,
        status INTEGER DEFAULT 0,
        test_high REAL,
        entry_date TEXT,
        update_time TEXT
    )
''')
conn.commit()

print("2. 正在通过 AkShare 获取全市场A股代码列表...")
df_all_stocks = ak.stock_info_a_code_name()
test_stocks = df_all_stocks.head(10)
print(f"成功获取代码表，准备拉取 {len(test_stocks)} 只股票。")

print("3. 登陆 BaoStock 系统...")
lg = bs.login() # 必须先登陆
if lg.error_code != '0':
    print(f"BaoStock 登陆失败: {lg.error_msg}")
else:
    print("BaoStock 登陆成功！开始批量拉取数据...")

    success_count = 0
    start_date = "2023-10-01" # BaoStock的日期格式带横杠

    for index, row in test_stocks.iterrows():
        code = row['code']
        name = row['name']
        
        # BaoStock 要求的代码格式是 sh.600000 或 sz.000001
        bs_code = f"sh.{code}" if code.startswith(('6', '9')) else f"sz.{code}"
        
        try:
            # 拉取历史K线，adjustflag="2" 代表前复权
            rs = bs.query_history_k_data_plus(
                bs_code,
                "date,code,open,high,low,close,volume,amount",
                start_date=start_date,
                frequency="d",
                adjustflag="2" 
            )
            
            if rs.error_code == '0':
                # 获取数据并转为 DataFrame
                data_list = []
                while (rs.error_code == '0') & rs.next():
                    data_list.append(rs.get_row_data())
                
                if data_list:
                    df_hist = pd.DataFrame(data_list, columns=rs.fields)
                    
                    # BaoStock 返回的数字是字符串格式，我们需要转换成浮点数
                    cols_to_convert = ['open', 'high', 'low', 'close', 'volume', 'amount']
                    df_hist[cols_to_convert] = df_hist[cols_to_convert].apply(pd.to_numeric)
                    
                    # 清理代码列，去掉 sh. 和 sz. 前缀，保持我们数据库的干净
                    df_hist['code'] = code 
                    
                    # 写入数据库
                    df_hist.to_sql(name='daily_k_line', con=conn, if_exists='append', index=False)
                    
                    cursor.execute('''
                        INSERT OR IGNORE INTO stock_pipeline (code, name, status, update_time)
                        VALUES (?, ?, 0, date('now'))
                    ''', (code, name))
                    conn.commit()
                    
                    success_count += 1
                    print(f"[{success_count}/10] {name} ({code}) 数据已入库 ✅")
                else:
                    print(f"警告：{name} ({code}) 没有获取到数据（可能是停牌）。")
            else:
                print(f"拉取 {name} ({code}) 失败，错误信息：{rs.error_msg}")
                
        except Exception as e:
            print(f"程序异常，拉取 {name} ({code}) 失败：{e}")

    # 收尾登出
    bs.logout()
    conn.close()
    print(f"\n🎉 终极建库任务完成！成功写入 {success_count} 只股票的历史数据。")

1. 数据库连接成功，正在创建表结构...
2. 正在通过 AkShare 获取全市场A股代码列表...


  0%|          | 0/16 [00:00<?, ?it/s]

成功获取代码表，准备拉取 10 只股票。
3. 登陆 BaoStock 系统...
login success!
BaoStock 登陆成功！开始批量拉取数据...
[1/10] 平安银行 (000001) 数据已入库 ✅
[2/10] 万  科Ａ (000002) 数据已入库 ✅
[3/10] *ST国华 (000004) 数据已入库 ✅
[4/10] 深振业Ａ (000006) 数据已入库 ✅
[5/10] 全新好 (000007) 数据已入库 ✅
[6/10] 神州高铁 (000008) 数据已入库 ✅
[7/10] 中国宝安 (000009) 数据已入库 ✅
[8/10] 美丽生态 (000010) 数据已入库 ✅
[9/10] 深物业A (000011) 数据已入库 ✅
[10/10] 南  玻Ａ (000012) 数据已入库 ✅
logout success!

🎉 终极建库任务完成！成功写入 10 只股票的历史数据。


In [2]:
import sqlite3
import pandas as pd

# 1. 连接我们刚刚建好的数据库
db_name = 'stock_quant.db'
conn = sqlite3.connect(db_name)

print("🔍 正在进行数据库健康度全面体检...\n")

# ==========================================
# 检查项 1：流水线状态表 (stock_pipeline)
# ==========================================
print("=== 检查项 1：状态机总览 ===")
df_pipeline = pd.read_sql_query("SELECT * FROM stock_pipeline", conn)
print(f"✅ 状态机中目前监控的股票总数: {len(df_pipeline)} 只")
print("前 5 只股票状态如下：")
print(df_pipeline.head().to_markdown(index=False))
print("\n")


# ==========================================
# 检查项 2：K线数据表整体概况 (daily_k_line)
# ==========================================
print("=== 检查项 2：历史数据池深度 ===")
cursor = conn.cursor()

# 查总行数
cursor.execute("SELECT COUNT(*) FROM daily_k_line")
total_rows = cursor.fetchone()[0]
print(f"✅ K线历史库总数据量: {total_rows} 条 (也就是 {total_rows} 根日K线)")

# 查时间跨度
cursor.execute("SELECT MIN(date), MAX(date) FROM daily_k_line")
min_date, max_date = cursor.fetchone()
print(f"✅ 数据时间跨度: 从 {min_date} 到 {max_date}")
print("\n")


# ==========================================
# 检查项 3：单只股票数据精度抽查
# ==========================================
print("=== 检查项 3：核心字段精度抽查 ===")
if not df_pipeline.empty:
    sample_code = df_pipeline.iloc[0]['code']
    sample_name = df_pipeline.iloc[0]['name']
    
    print(f"正在抽查: {sample_name} ({sample_code}) 的最新 5 天数据...")
    # 取这只股票按日期倒序（最新的）5条数据
    df_sample = pd.read_sql_query(
        f"SELECT date, code, open, high, low, close, volume FROM daily_k_line WHERE code='{sample_code}' ORDER BY date DESC LIMIT 5", 
        conn
    )
    print(df_sample.to_markdown(index=False))
    
    # 数据合法性基本校验
    if df_sample['close'].isnull().any() or df_sample['volume'].isnull().any():
        print("\n❌ 警告：抽查发现存在空值 (NaN)，数据可能有残缺！")
    elif (df_sample['high'] < df_sample['low']).any():
        print("\n❌ 警告：抽查发现有最高价低于最低价的极其离谱的数据，请查验！")
    else:
        print("\n✅ 数据精度校验通过！未发现空值，且高低价逻辑正常。")

# 关闭连接
conn.close()

🔍 正在进行数据库健康度全面体检...

=== 检查项 1：状态机总览 ===
✅ 状态机中目前监控的股票总数: 10 只
前 5 只股票状态如下：
|   code | name     |   status | test_high   | entry_date   | update_time   |
|-------:|:---------|---------:|:------------|:-------------|:--------------|
| 000001 | 平安银行 |        0 |             |              | 2026-04-10    |
| 000002 | 万  科Ａ |        0 |             |              | 2026-04-10    |
| 000004 | *ST国华  |        0 |             |              | 2026-04-10    |
| 000006 | 深振业Ａ |        0 |             |              | 2026-04-10    |
| 000007 | 全新好   |        0 |             |              | 2026-04-10    |


=== 检查项 2：历史数据池深度 ===
✅ K线历史库总数据量: 6080 条 (也就是 6080 根日K线)
✅ 数据时间跨度: 从 2023-10-09 到 2026-04-10


=== 检查项 3：核心字段精度抽查 ===
正在抽查: 平安银行 (000001) 的最新 5 天数据...
| date       |   code |   open |   high |   low |   close |   volume |
|:-----------|-------:|-------:|-------:|------:|--------:|---------:|
| 2026-04-10 | 000001 |  11.1  |  11.13 | 11.07 |   11.09 | 48104693 |
| 2026-04-09 | 000001 |  11

In [3]:
import sqlite3
import pandas as pd

# 1. 连接数据库
conn = sqlite3.connect('stock_quant.db')

# ==========================================
# 动作 1：读取并打印“股票名单和状态表”
# ==========================================
print("【表1：股票流水线状态 (stock_pipeline)】")
df_pipeline = pd.read_sql("SELECT * FROM stock_pipeline", conn)
print(df_pipeline)
print("-" * 50)

# ==========================================
# 动作 2：读取历史K线数据，并导出 Excel
# ==========================================
# 获取名单里的第一只股票代码（比如万科A 000002），或者你也可以自己指定
target_code = df_pipeline['code'].iloc[0] if not df_pipeline.empty else "000001"
print(f"【表2：正在提取 {target_code} 的历史K线数据...】")

# 从数据库提取这只股票的所有历史数据，按日期倒序（最新的在上面）
query = f"SELECT * FROM daily_k_line WHERE code='{target_code}' ORDER BY date DESC"
df_kline = pd.read_sql(query, conn)

# 在终端只打印最新的 5 天数据给你看一眼
print(df_kline.head())

# 核心步骤：直接导出为 Excel 文件，方便你手动慢慢比对！
export_filename = f"对比数据_{target_code}.xlsx"
df_kline.to_excel(export_filename, index=False)

conn.close()

print("-" * 50)
print(f"✅ 搞定！完整数据已保存到当前文件夹的【{export_filename}】文件中。")
print("快去左侧文件目录双击下载或打开它，对照着同花顺/通达信的 K 线图比对一下精度吧！")

【表1：股票流水线状态 (stock_pipeline)】
     code   name  status test_high entry_date update_time
0  000001   平安银行       0      None       None  2026-04-10
1  000002  万  科Ａ       0      None       None  2026-04-10
2  000004  *ST国华       0      None       None  2026-04-10
3  000006   深振业Ａ       0      None       None  2026-04-10
4  000007    全新好       0      None       None  2026-04-10
5  000008   神州高铁       0      None       None  2026-04-10
6  000009   中国宝安       0      None       None  2026-04-10
7  000010   美丽生态       0      None       None  2026-04-10
8  000011   深物业A       0      None       None  2026-04-10
9  000012  南  玻Ａ       0      None       None  2026-04-10
--------------------------------------------------
【表2：正在提取 000001 的历史K线数据...】
         date    code   open   high    low  close    volume        amount
0  2026-04-10  000001  11.10  11.13  11.07  11.09  48104693  5.340890e+08
1  2026-04-09  000001  11.17  11.22  11.06  11.10  60236493  6.692393e+08
2  2026-04-08  000001  11.11  1

In [1]:
import akshare as ak
import baostock as bs
import pandas as pd
import sqlite3
import time
from datetime import datetime

# ==========================================
# 核心配置区
# ==========================================
db_name = 'stock_quant.db'
start_date = "2023-01-01" # 建议拉取最近一年多的数据，足够计算60日/120日横盘
end_date = datetime.now().strftime("%Y-%m-%d")

# 1. 连接数据库
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# 确保表结构存在
cursor.execute('''
    CREATE TABLE IF NOT EXISTS stock_pipeline (
        code TEXT PRIMARY KEY,
        name TEXT,
        status INTEGER DEFAULT 0,
        test_high REAL,
        entry_date TEXT,
        update_time TEXT
    )
''')
conn.commit()

print("1. 正在获取全市场 A 股代码列表...")
df_all_stocks = ak.stock_info_a_code_name()
total_stocks = len(df_all_stocks)
print(f"✅ 获取成功！当前全市场共有 {total_stocks} 只股票。")

# ==========================================
# 【断点续传核心逻辑】检查数据库里已经有哪些股票了
# ==========================================
cursor.execute("SELECT code FROM stock_pipeline")
processed_codes = set([row[0] for row in cursor.fetchall()])
print(f"📦 数据库中已存在 {len(processed_codes)} 只股票的数据。")

stocks_to_process = df_all_stocks[~df_all_stocks['code'].isin(processed_codes)]
remain_count = len(stocks_to_process)
print(f"⏳ 本次实际需要拉取 {remain_count} 只股票。\n")

if remain_count == 0:
    print("🎉 所有股票数据已全部下载完毕，无需重复执行！")
else:
    print("2. 登陆 BaoStock 系统，开始全量拉取...")
    lg = bs.login()
    
    if lg.error_code != '0':
        print(f"❌ BaoStock 登陆失败: {lg.error_msg}")
    else:
        success_count = 0
        empty_count = 0
        error_count = 0

        # 开始遍历需要拉取的股票
        for index, row in stocks_to_process.iterrows():
            code = row['code']
            name = row['name']
            
            # BaoStock 市场前缀规则：6开头是沪市(sh)，0和3开头是深市(sz)，4和8开头是北交所(bj)
            if code.startswith('6'):
                bs_code = f"sh.{code}"
            elif code.startswith(('0', '3')):
                bs_code = f"sz.{code}"
            else:
                bs_code = f"bj.{code}"
            
            try:
                # 请求 K 线数据
                rs = bs.query_history_k_data_plus(
                    bs_code,
                    "date,code,open,high,low,close,volume,amount",
                    start_date=start_date,
                    end_date=end_date,
                    frequency="d",
                    adjustflag="2" # 2代表前复权
                )
                
                if rs.error_code == '0':
                    data_list = []
                    while (rs.error_code == '0') & rs.next():
                        data_list.append(rs.get_row_data())
                    
                    if data_list:
                        # 有数据，写入数据库
                        df_hist = pd.DataFrame(data_list, columns=rs.fields)
                        cols_to_convert = ['open', 'high', 'low', 'close', 'volume', 'amount']
                        df_hist[cols_to_convert] = df_hist[cols_to_convert].apply(pd.to_numeric)
                        df_hist['code'] = code 
                        
                        # 写入 K 线表
                        df_hist.to_sql(name='daily_k_line', con=conn, if_exists='append', index=False)
                        
                        # 写入状态表 (只要写入了状态表，下次运行就会跳过它)
                        cursor.execute('''
                            INSERT OR IGNORE INTO stock_pipeline (code, name, status, update_time)
                            VALUES (?, ?, 0, date('now'))
                        ''', (code, name))
                        conn.commit()
                        
                        success_count += 1
                    else:
                        # 没有数据（可能是刚上市、长期停牌或已退市）
                        # 同样写入状态表，但可以标记一个特殊状态（例如 99 代表无数据），避免下次重复去拉
                        cursor.execute('''
                            INSERT OR IGNORE INTO stock_pipeline (code, name, status, update_time)
                            VALUES (?, ?, 99, date('now'))
                        ''', (code, name))
                        conn.commit()
                        empty_count += 1
                else:
                    print(f"⚠️ 拉取 {name} ({code}) 失败: {rs.error_msg}")
                    error_count += 1
                    
                # 打印进度条（每拉取 50 只打印一次，避免屏幕滚动太快）
                current_processed = success_count + empty_count + error_count
                if current_processed % 50 == 0 or current_processed == remain_count:
                    percent = (current_processed / remain_count) * 100
                    print(f"🚀 进度: {current_processed}/{remain_count} ({percent:.2f}%) - 最新入库: {name}")

            except Exception as e:
                print(f"❌ 严重异常: {name} ({code}) - {e}")
                error_count += 1

        bs.logout()
        print("\n" + "="*50)
        print("🎉 全量数据拉取任务结束！")
        print(f"📊 成功入库: {success_count} 只")
        print(f"🈳 无数据(停牌/退市): {empty_count} 只")
        print(f"⚠️ 报错失败: {error_count} 只")
        print("="*50)

# 关闭数据库连接
conn.close()

1. 正在获取全市场 A 股代码列表...


  0%|          | 0/16 [00:00<?, ?it/s]

✅ 获取成功！当前全市场共有 5502 只股票。
📦 数据库中已存在 5197 只股票的数据。
⏳ 本次实际需要拉取 305 只股票。

2. 登陆 BaoStock 系统，开始全量拉取...
login success!
⚠️ 拉取 安徽凤凰 (920000) 失败: 股票代码未标识sh或sz
⚠️ 拉取 纬达光电 (920001) 失败: 股票代码未标识sh或sz
⚠️ 拉取 万达轴承 (920002) 失败: 股票代码未标识sh或sz
⚠️ 拉取 中诚咨询 (920003) 失败: 股票代码未标识sh或sz
⚠️ 拉取 鼎佳精密 (920005) 失败: 股票代码未标识sh或sz
⚠️ 拉取 晟楠科技 (920006) 失败: 股票代码未标识sh或sz
⚠️ 拉取 酉立智能 (920007) 失败: 股票代码未标识sh或sz
⚠️ 拉取 成电光信 (920008) 失败: 股票代码未标识sh或sz
⚠️ 拉取 丹娜生物 (920009) 失败: 股票代码未标识sh或sz
⚠️ 拉取 凯添燃气 (920010) 失败: 股票代码未标识sh或sz
⚠️ 拉取 晨光电机 (920011) 失败: 股票代码未标识sh或sz
⚠️ 拉取 特瑞斯 (920014) 失败: 股票代码未标识sh或sz
⚠️ 拉取 锦华新材 (920015) 失败: 股票代码未标识sh或sz
⚠️ 拉取 中草香料 (920016) 失败: 股票代码未标识sh或sz
⚠️ 拉取 星昊医药 (920017) 失败: 股票代码未标识sh或sz
⚠️ 拉取 宏远股份 (920018) 失败: 股票代码未标识sh或sz
⚠️ 拉取 铜冠矿建 (920019) 失败: 股票代码未标识sh或sz
⚠️ 拉取 泰凯英 (920020) 失败: 股票代码未标识sh或sz
⚠️ 拉取 流金科技 (920021) 失败: 股票代码未标识sh或sz
⚠️ 拉取 世昌股份 (920022) 失败: 股票代码未标识sh或sz
⚠️ 拉取 田野股份 (920023) 失败: 股票代码未标识sh或sz
⚠️ 拉取 卓兆点胶 (920026) 失败: 股票代码未标识sh或sz
⚠️ 拉取 交大铁发 (920027) 失败: 股票代码未标识sh或sz
⚠️ 拉取 新恒泰 (920028) 失败: 股票代码未标识sh或sz
⚠️ 

In [5]:
import sqlite3
import pandas as pd
import numpy as np

print("⚙️ 正在启动策略引擎，加载历史数据...")

# 1. 连接数据库并加载数据
conn = sqlite3.connect('stock_quant.db')

# 为了计算 60 日横盘和 5 日均量，我们不需要全部历史数据
# 这里我们巧妙地使用 SQL 只提取每只股票最近 100 个交易日的数据，极大地节省内存和提速
query = """
SELECT code, date, open, high, low, close, volume 
FROM daily_k_line 
WHERE date >= date('now', '-5 months')
"""
df = pd.read_sql(query, conn)

print(f"📦 成功加载 {len(df)} 条近期 K 线数据，准备进行向量化计算...")

# 2. 数据预处理与排序 (极其重要：按股票代码和时间正序排列)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['code', 'date'])

# ==========================================
# 3. 核心指标计算 (Pandas 向量化极速运算)
# ==========================================
print("🧮 正在计算 60 日振幅、5 日均量等核心策略指标...")

# 计算 5日均量
df['vol_ma5'] = df.groupby('code')['volume'].transform(lambda x: x.rolling(window=5).mean())

# 计算过去 60 个交易日的最高价和最低价
df['high_60'] = df.groupby('code')['high'].transform(lambda x: x.rolling(window=60).max())
df['low_60'] = df.groupby('code')['low'].transform(lambda x: x.rolling(window=60).min())

# 计算 60 日横盘振幅：(60日最高 - 60日最低) / 60日最低
df['amplitude_60'] = (df['high_60'] - df['low_60']) / df['low_60']

# 获取每只股票【最新的一天】的数据进行策略判断
# drop_duplicates(subset=['code'], keep='last') 会自动保留每只股票的最后一行（即最新交易日）
df_latest = df.drop_duplicates(subset=['code'], keep='last').copy()

# ==========================================
# 4. 漏斗筛选：执行“倍量试盘”策略逻辑
# ==========================================
print("🔎 正在执行策略漏斗筛选...")

# 【条件 A】：大前提必须是长期横盘（比如 60 日振幅小于 30%）
# 你可以根据实战经验，调整 0.30 这个参数
cond_sideways = df_latest['amplitude_60'] < 0.30

# 【条件 B】：今天的成交量，必须大于过去 5 日均量的 2 倍（倍量）
cond_double_vol = df_latest['volume'] > (df_latest['vol_ma5'].shift(1) * 2) 

# 【条件 C】：今天是收阳线的（收盘价 > 开盘价），且价格最好有所突破或异动
# 这里我们加一个基础条件：收盘价相比 60日最低价，涨幅不能超过 40%（防止已经在高位了）
cond_bottom_area = (df_latest['close'] - df_latest['low_60']) / df_latest['low_60'] < 0.40
cond_yang = df_latest['close'] > df_latest['open']

# 综合以上所有条件，找出今天刚刚符合“倍量试盘”的标的
test_pool = df_latest[cond_sideways & cond_double_vol & cond_bottom_area & cond_yang]

print("-" * 50)
print(f"🎯 筛选完毕！今日触发【倍量试盘】信号的股票共有: {len(test_pool)} 只")

# ==========================================
# 5. 更新状态机数据库，并导出名单
# ==========================================
if not test_pool.empty:
    cursor = conn.cursor()
    
    # 提取符合条件的股票代码列表
    target_codes = test_pool['code'].tolist()
    
    # 将这些股票在流水线表中的状态升级为 1 (Testing 试盘状态)
    # 并记录试盘当天的最高价，作为日后突破的观察点
    for index, row in test_pool.iterrows():
        cursor.execute(f"""
            UPDATE stock_pipeline 
            SET status = 1, 
                test_high = {row['high']}, 
                update_time = date('now')
            WHERE code = '{row['code']}'
        """)
    conn.commit()
    print("💾 数据库状态机更新完成！已将标的移入【试盘组】。")
    
    # 导出 Excel 方便查看
    output_cols = ['code', 'date', 'close', 'volume', 'vol_ma5', 'amplitude_60']
    test_pool[output_cols].to_excel('今日倍量试盘名单.xlsx', index=False)
    print("📊 详细名单已导出为『今日倍量试盘名单.xlsx』，快去左侧打开看看吧！")
else:
    print("🤷‍♂️ 今天市场平淡，没有符合『横盘且倍量试盘』严格条件的股票。")

conn.close()

⚙️ 正在启动策略引擎，加载历史数据...
📦 成功加载 517934 条近期 K 线数据，准备进行向量化计算...
🧮 正在计算 60 日振幅、5 日均量等核心策略指标...
🔎 正在执行策略漏斗筛选...
--------------------------------------------------
🎯 筛选完毕！今日触发【倍量试盘】信号的股票共有: 165 只
💾 数据库状态机更新完成！已将标的移入【试盘组】。
📊 详细名单已导出为『今日倍量试盘名单.xlsx』，快去左侧打开看看吧！
